# Linear Regression (cross-validation)
One of three algorithms compared. Uses the same split, preprocessing, CV folds and primary metric (RMSE) as notebooks 02 and 03.
Linear Regression has no hyperparameters to tune. The test set is not touched here.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

# Works whether the notebook is run from the repo root or from notebooks/
DATA_PATH = next(
    p for p in [Path("data/original/calorie_dataset.csv"),
                Path("../data/original/calorie_dataset.csv"),
                Path("calorie_dataset.csv")]
    if p.exists()
)
df = pd.read_csv(DATA_PATH)

FEATURES = ["protein_g", "carbs_g", "fat_g", "fiber_g", "sugar_g"]
TARGET = "calories"        

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, "| Test:", X_test.shape, "(test set is NOT used in this notebook)")

Train: (4000, 5) | Test: (1000, 5) (test set is NOT used in this notebook)


In [ ]:

preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), FEATURES)])
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    "rmse": "neg_root_mean_squared_error",   
    "mae":  "neg_mean_absolute_error",
    "r2":   "r2",
}

def summarize_cv(model, name):
    """5-fold CV on the TRAINING set only; prints mean +/- std for RMSE, MAE, R2."""
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=SCORING)
    rmse, mae, r2 = -res["test_rmse"], -res["test_mae"], res["test_r2"]
    print(name)
    print(f"CV RMSE: {rmse.mean():.4f} +/- {rmse.std():.4f}")
    print(f"CV MAE : {mae.mean():.4f} +/- {mae.std():.4f}")
    print(f"CV R2  : {r2.mean():.4f} +/- {r2.std():.4f}")
    return {"model": name, "rmse_mean": rmse.mean(), "rmse_std": rmse.std(),
            "mae_mean": mae.mean(), "mae_std": mae.std(),
            "r2_mean": r2.mean(), "r2_std": r2.std()}

In [3]:
from sklearn.linear_model import LinearRegression

lr_pipe = Pipeline([
    ("pre", preprocessor),
    ("model", LinearRegression())
])
lr_results = summarize_cv(lr_pipe, "Linear Regression")

Linear Regression
CV RMSE: 30.0393 +/- 0.3842
CV MAE : 23.9103 +/- 0.4132
CV R2  : 0.9357 +/- 0.0026
